In [24]:
import hashlib
import json
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(Path(".env"), override=True)

TOKEN = os.environ.get("TMDB_TOKEN", "").strip()
if not TOKEN:
    raise RuntimeError("TMDB_TOKEN is missing from .env")

BASE = "https://api.themoviedb.org/3"
SEARCH = BASE + "/discover/movie"
HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}
START_DATE = "2015-01-01"
END_DATE = "2026-10-04"
PARAMS = {
    "with_original_language": "en",
    "primary_release_date.gte": START_DATE,
    "primary_release_date.lte": END_DATE,
    "vote_count.gte": 100,
    "include_adult": "false",
    "include_video": "false",
    "language": "en-US",
    "sort_by": "primary_release_date.asc",
}
CACHE = Path("cache_tmdb")
CACHE.mkdir(exist_ok=True)
OUT = Path("tmdb_m1")
OUT.mkdir(exist_ok=True)
POLITE_DELAY = 0.25  
http_attempts = 0
cache_hits = 0
cache_used = set()
started_at = datetime.now(timezone.utc).isoformat()


In [25]:
def http_get(url, params=None, headers=None, attempts=5):
    global http_attempts
    if headers is None:
        headers = HEADERS
    for k in range(attempts):
        http_attempts += 1
        try:
            r = requests.get(url, params=params, headers=headers, timeout=10)
        except requests.RequestException as exc:
            status, retry_after = type(exc).__name__, 0
        else:
            if r.status_code == 200:
                return r.json()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()
                raise RuntimeError(f"Unexpected HTTP status {r.status_code}: {url}")
            status = r.status_code
            try:
                retry_after = int(r.headers.get("Retry-After", 0))
            except (ValueError, TypeError):
                retry_after = 0
        if k + 1 < attempts:
            wait = retry_after if retry_after > 0 else random.uniform(0, 2**k)
            print(f"{status} on attempt {k + 1}; waiting {wait:.1f}s")
            time.sleep(wait)
    raise RuntimeError(f"Gave up after {attempts} attempts: {url}")


In [26]:
def cache_key(url, params=None):
    payload = url + "|" + json.dumps(params or {}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


def cached_get(url, params=None):
    global cache_hits
    path = CACHE / f"{cache_key(url, params)}.json"
    cache_used.add(path)
    if path.exists():
        cache_hits += 1
        return json.loads(path.read_text(encoding="utf-8"))
    body = http_get(url, params)
    path.write_text(json.dumps(body), encoding="utf-8")
    time.sleep(POLITE_DELAY)
    return body


In [27]:
log = {
    "source": "TMDB", "api_version": "3",
    "scope": f"English-original-language movies, {START_DATE} to {END_DATE}, at least 100 ratings; adult and video entries excluded",
    "scope_revision": None, 
    "started_at_utc": started_at,
    "endpoint": SEARCH, "parameters": PARAMS,
    "row_units": {"movies_raw.parquet": "one unique movie ID", "movie_genres.parquet": "one movie-genre association"},
    "complete": False,
}
movies = []
pages_fetched = 0
try:
    page = 1
    while True:
        payload = cached_get(SEARCH, {**PARAMS, "page": page})
        if page == 1:
            total_results = int(payload["total_results"])
            total_pages = int(payload["total_pages"])
            log.update(reported_total_results=total_results, reported_total_pages=total_pages)
            if total_pages > 500:
                raise RuntimeError("More than 500 pages: split the scope into smaller date windows.")
        results = payload["results"]
        pages_fetched += 1
        movies.extend(results)
        print(f"Page {page}/{total_pages}: {len(results)} movies; {len(movies)} collected")
        if not results and page < total_pages:
            raise RuntimeError("Unexpected empty page before the reported last page.")
        if page >= total_pages:
            break
        page += 1
    genre_payload = cached_get(BASE + "/genre/movie/list", {"language": "en-US"})
    log["genre_endpoint"] = BASE + "/genre/movie/list"
except Exception as exc:
    log["error"] = f"{type(exc).__name__}: {exc}"
    raise
finally:
    log.update(pages_fetched=pages_fetched, records_received=len(movies),
               http_attempts=http_attempts, cache_hits=cache_hits,
               acquisition_finished_at_utc=datetime.now(timezone.utc).isoformat())
    log["cache_files_used"] = [
        {"file": str(p), "modified_at_utc": datetime.fromtimestamp(p.stat().st_mtime, timezone.utc).isoformat()}
        for p in sorted(cache_used) if p.exists()
    ]
    (OUT / "acquisition_log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")

print("Reported:", total_results, "Received:", len(movies))


Page 1/273: 20 movies; 20 collected
Page 2/273: 20 movies; 40 collected
Page 3/273: 20 movies; 60 collected
Page 4/273: 20 movies; 80 collected
Page 5/273: 20 movies; 100 collected
Page 6/273: 20 movies; 120 collected
Page 7/273: 20 movies; 140 collected
Page 8/273: 20 movies; 160 collected
Page 9/273: 20 movies; 180 collected
Page 10/273: 20 movies; 200 collected
Page 11/273: 20 movies; 220 collected
Page 12/273: 20 movies; 240 collected
Page 13/273: 20 movies; 260 collected
Page 14/273: 20 movies; 280 collected
Page 15/273: 20 movies; 300 collected
Page 16/273: 20 movies; 320 collected
Page 17/273: 20 movies; 340 collected
Page 18/273: 20 movies; 360 collected
Page 19/273: 20 movies; 380 collected
Page 20/273: 20 movies; 400 collected
Page 21/273: 20 movies; 420 collected
Page 22/273: 20 movies; 440 collected
Page 23/273: 20 movies; 460 collected
Page 24/273: 20 movies; 480 collected
Page 25/273: 20 movies; 500 collected
Page 26/273: 20 movies; 520 collected
Page 27/273: 20 movies; 5

In [28]:
if not movies:
    raise RuntimeError("No movies collected; check your scope and API response.")
df = pd.json_normalize(movies)
df = df.drop(columns=["adult", "video", "poster_path", "backdrop_path"])
duplicates = int(df["id"].duplicated().sum())
df = df.drop_duplicates(subset="id").reset_index(drop=True)

assert df["original_language"].eq("en").all()
assert df["vote_count"].ge(100).all()
assert df["release_date"].between(START_DATE, END_DATE).all()

genre_names = {g["id"]: g["name"] for g in genre_payload["genres"]}
movie_genres = (df[["id", "genre_ids"]]
                .explode("genre_ids")
                .dropna(subset=["genre_ids"])
                .rename(columns={"id": "movie_id", "genre_ids": "genre_id"}))
movie_genres["genre_id"] = movie_genres["genre_id"].astype("int64")
movie_genres["genre_name"] = movie_genres["genre_id"].map(genre_names)
movie_genres = movie_genres.reset_index(drop=True)

print("Movie rows:", df.shape)
print("Movie-genre rows:", movie_genres.shape)
print("Duplicate movie IDs removed:", duplicates)
print("Missing values:")
print(df.isna().sum())
df.head(3)


Movie rows: (5455, 11)
Movie-genre rows: (13696, 3)
Duplicate movie IDs removed: 0
Missing values:
genre_ids            0
id                   0
title                0
original_language    0
original_title       0
overview             0
popularity           0
release_date         0
softcore             0
vote_average         0
vote_count           0
dtype: int64


,genre_ids,id,title,original_language,original_title,overview,popularity,release_date,softcore,vote_average,vote_count
0,[18],320367,Return,en,Return,A tale of terror. Cathy Reed has been institut...,0.8235,2015-01-01,False,6.600,1177
1,"[10749, 35]",313867,Roald Dahl's Esio Trot,en,Roald Dahl's Esio Trot,Retired bachelor Mr Hoppy is hopelessly in lov...,3.2243,2015-01-02,False,6.700,128
2,[27],299551,The Atticus Institute,en,The Atticus Institute,"In the early 1970s, Dr. Henry West creates an ...",3.1234,2015-01-05,False,5.759,373


In [29]:
df.to_parquet(OUT / "movies_raw.parquet", index=False)
movie_genres.to_parquet(OUT / "movie_genres.parquet", index=False)
back = pd.read_parquet(OUT / "movies_raw.parquet")
genres_back = pd.read_parquet(OUT / "movie_genres.parquet")
assert back.shape == df.shape
assert genres_back.shape == movie_genres.shape
assert back["id"].tolist() == df["id"].tolist()

def list_value(value):
    if value is None:
        return None
    if hasattr(value, "__len__") and not isinstance(value, str):
        return list(value)
    if pd.isna(value):
        return None
    raise TypeError("Unexpected value in genre_ids")

assert [list_value(v) for v in df["genre_ids"]] == [list_value(v) for v in back["genre_ids"]]
assert df["genre_ids"].isna().sum() == back["genre_ids"].isna().sum()
pd.testing.assert_frame_equal(movie_genres, genres_back)
print("Shapes, IDs, genre lists and genre associations survived the round-trip.")
print("Returned genre_ids type:", type(back["genre_ids"].iloc[0]).__name__)

shapes = {"movies_raw.parquet": list(back.shape), "movie_genres.parquet": list(genres_back.shape)}
log.update(
    complete=True,
    parquet_shapes=shapes,
    unique_movies=len(df),
    duplicate_movie_ids_removed=duplicates,
    received_count_matches_reported=(len(movies) == total_results),
    unique_count_matches_reported=(len(df) == total_results),
    round_trip_checks_passed=True,
    personal_data="Movie metadata only; no people profiles or user-level ratings collected.",
    snapshot_caveat="Page-based API results can change while downloading. Reused cache files may have different timestamps; see cache_files_used.",
)
(OUT / "acquisition_log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
shape_text = "\n".join(f"{name}: {shape[0]} rows, {shape[1]} columns" for name, shape in shapes.items())
(OUT / "parquet_shapes.txt").write_text(shape_text + "\n", encoding="utf-8")
print(shape_text)
print("M1 outputs are in:", OUT.resolve())


Shapes, IDs, genre lists and genre associations survived the round-trip.
Returned genre_ids type: ndarray
movies_raw.parquet: 5455 rows, 11 columns
movie_genres.parquet: 13696 rows, 3 columns
M1 outputs are in: /Users/aminenakari/Downloads/M1 AI/Data Aq./Project/Data Acq Project/Data-Acq-Project/tmdb_m1
